# Transcribe Pipeline
Google Colab pipeline: Google Drive -> Whisper large-v3 -> Pyannote diarization -> TXT/JSON/SRT.
Heavy inference runs on the selected Colab GPU.


In [ ]:
!pip -q install -U faster-whisper pyannote.audio==3.1.1 torch torchaudio

from google.colab import drive, userdata
from pathlib import Path
import json, os, re, time, torch

drive.mount('/content/drive')
ROOT=Path('/content/drive/MyDrive/transcribe'); INPUT=ROOT/'input'; OUTPUT=ROOT/'output'; STATE=ROOT/'state'
for p in (INPUT, OUTPUT, STATE): p.mkdir(parents=True, exist_ok=True)

if not torch.cuda.is_available():
    raise RuntimeError('CUDA GPU is required. In Colab select Runtime > Change runtime type > GPU.')
gpu=torch.cuda.get_device_name(0)
print('GPU:', gpu)
if not any(x in gpu.upper() for x in ('T4','V100','A100','L4')): print('WARNING: unexpected GPU type; pipeline can still continue.')

try:
    HF_TOKEN=userdata.get('HUGGINGFACE_TOKEN')
except Exception as exc:
    raise RuntimeError('Create the Colab Secret HUGGINGFACE_TOKEN and allow notebook access.') from exc
if not HF_TOKEN: raise RuntimeError('HUGGINGFACE_TOKEN is empty.')


In [ ]:
from faster_whisper import WhisperModel
from pyannote.audio import Pipeline

whisper=WhisperModel('large-v3', device='cuda', compute_type='float16')
diarizer=Pipeline.from_pretrained('pyannote/speaker-diarization-3.1', use_auth_token=HF_TOKEN)
diarizer.to(torch.device('cuda'))
print('Models loaded on GPU')


In [ ]:
registry=STATE/'processed.json'
try:
    processed=json.loads(registry.read_text(encoding='utf-8')) if registry.exists() else {}
    if not isinstance(processed, dict): processed={}
except (OSError, json.JSONDecodeError):
    processed={}

audio_ext={'.wav','.mp3','.m4a','.flac','.ogg','.opus','.aac','.webm','.mp4'}

def stamp(seconds, comma=False):
    ms=max(0,int(round(float(seconds)*1000))); h,ms=divmod(ms,3600000); m,ms=divmod(ms,60000); s,ms=divmod(ms,1000)
    sep=',' if comma else '.'
    return f'{h:02d}:{m:02d}:{s:02d}{sep}{ms:03d}' if comma else f'{h:02d}:{m:02d}:{s:02d}'

def assign_speaker(start,end,turns):
    scores={}
    for a,b,label in turns:
        overlap=max(0.0,min(end,b)-max(start,a))
        if overlap>0: scores[label]=scores.get(label,0.0)+overlap
    return max(scores,key=scores.get) if scores else 'UNKNOWN'

def atomic_write(path,text):
    tmp=path.with_suffix(path.suffix+'.tmp'); tmp.write_text(text,encoding='utf-8'); tmp.replace(path)

files=[p for p in sorted(INPUT.rglob('*')) if p.is_file() and p.suffix.lower() in audio_ext]
pending=[p for p in files if str(p.relative_to(INPUT)) not in processed]
print(f'Found {len(files)} audio files; pending: {len(pending)}')


In [ ]:
for audio in pending:
    key=str(audio.relative_to(INPUT))
    print(f'\nProcessing: {key}')
    try:
        diar=diarizer(str(audio))
        turns=[(s.start,s.end,sp) for s,_,sp in diar.itertracks(yield_label=True)]
        segments,_=whisper.transcribe(str(audio), beam_size=5, vad_filter=True, word_timestamps=False)
        rows=[]
        for seg in segments:
            text=seg.text.strip()
            if text:
                rows.append({'start':float(seg.start),'end':float(seg.end),'speaker':assign_speaker(seg.start,seg.end,turns),'text':text})
        target=(OUTPUT/key).with_suffix(''); target.parent.mkdir(parents=True,exist_ok=True)
        txt='\n'.join(f'[{stamp(r["start"])} - {r["speaker"]}]: {r["text"]}' for r in rows)+'\n'
        atomic_write(target.with_suffix('.txt'),txt)
        payload={'source':key,'model':'openai/whisper-large-v3','diarization':'pyannote/speaker-diarization-3.1','segments':rows}
        atomic_write(target.with_suffix('.json'),json.dumps(payload,ensure_ascii=False,indent=2))
        srt=[]
        for i,r in enumerate(rows,1):
            srt.append(f'{i}\n{stamp(r["start"],True)} --> {stamp(r["end"],True)}\n[{r["speaker"]}] {r["text"]}\n')
        atomic_write(target.with_suffix('.srt'),'\n'.join(srt))
        processed[key]={'completed_at':time.strftime('%Y-%m-%dT%H:%M:%SZ',time.gmtime()),'segments':len(rows)}
        atomic_write(registry,json.dumps(processed,ensure_ascii=False,indent=2))
        print(f'DONE: {key} ({len(rows)} segments)')
    except Exception as exc:
        print(f'ERROR: {key}: {exc}')

print('Queue finished. Failed files remain unregistered and can be retried.')


## Optional Colab keep-alive
Colab sessions are managed by Google and no browser snippet can guarantee indefinite runtime. If you choose to use a keep-alive snippet, run it only in your own browser console and only while the notebook is actively doing useful work. Never automate CAPTCHA or bypass access controls.


In [ ]:
# Browser-console helper (paste manually into DevTools Console, if permitted by your environment):
# setInterval(() => { document.querySelector('colab-connect-button')?.click(); }, 60000);
print('Keep-alive is intentionally manual; Google may ignore or change this UI.')
